# 04. 정규화 방법 비교 — MinMaxScaler vs Log Base

## 실험 배경

실험 1(MinMaxScaler)의 FAISS 평가 결과, Top-10 중 같은 종목 패턴이 평균 2.20개 포함되었다. 이는 MinMaxScaler가 절대 가격 수준을 제거하지만 **변동 비율을 구분하지 못하는 문제** 때문이다.

예: 1→6 (+600% 상승)과 10→60 (+500% 상승)이 정규화 후 거의 동일한 값을 갖는다.

이 노트북에서는 4가지 정규화 방법을 합성 패턴 3개로 비교해, 실험 3(Log Base)의 전처리 방식을 선택한 근거를 제시한다.

## 비교 대상

| 방법 | 수식 | 핵심 특성 |
|---|---|---|
| 1. MinMaxScaler | [min, max] → [0, 1] | 절대값 제거, 비율 구분 불가 (현재 실험 1) |
| 2. First Close 기준 | P_t / P_0 | 비율 반영, 범위 [0, ∞) 불안정 |
| 3. First Close + Clipping | clip(P_t/P_0, 0.5, 2.0) → [0, 1] | 비율 반영, ±100% 초과 변동은 잘림 |
| 4. Log Base | log(P_t / P_0) → MinMax | 비율 반영, 대칭, 금융 표준 (실험 3 선택) |

## 1. 라이브러리 임포트

numpy, pandas, matplotlib, MinMaxScaler를 불러온다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 2. 테스트 패턴 생성

비교 실험에 쓸 합성 패턴 3개를 만든다.

- **패턴 1**: 약한 상승 (1→6, +600%)
- **패턴 2**: 같은 모양의 강한 상승 (10→58, +580%) — 패턴 1과 동일 형태
- **패턴 3**: V자 패턴 (하락 후 반등) — 완전히 다른 형태

이상적인 정규화라면 패턴 1과 2는 동일한 값을, 패턴 3은 다른 값을 가져야 한다.

---

## 3. 정규화 함수 정의

4가지 정규화 함수를 정의한다. MinMaxScaler, 첫 Close 기준, 첫 Close + Clipping, Log Base 4가지를 각각 구현해 같은 패턴에 적용하고 결과를 비교한다.

In [ ]:
# 패턴 1: 약한 상승 (1 → 6, 600% 상승)
pattern1 = np.array([
    [1.0, 1.2, 0.9, 1.1],
    [1.1, 1.5, 1.0, 1.4],
    [1.4, 1.8, 1.3, 1.7],
    [1.7, 2.2, 1.6, 2.0],
    [2.0, 2.5, 1.9, 2.3],
    [2.3, 2.8, 2.2, 2.6],
    [2.6, 3.2, 2.5, 3.0],
    [3.0, 3.6, 2.9, 3.4],
    [3.4, 4.0, 3.3, 3.8],
    [3.8, 4.5, 3.7, 4.2],
    [4.2, 5.0, 4.1, 4.7],
    [4.7, 6.0, 4.6, 5.8],
])

# 패턴 2: 강한 상승 (10 → 58, 580% 상승, 같은 모양)
pattern2 = pattern1 * 10  # 같은 모양, 10배 스케일

# 패턴 3: V자 패턴 (하락 → 반등, 명확히 다른 모양!)
pattern3 = np.array([
    [50.0, 52.0, 48.0, 50.0],   # 50 (시작)
    [50.0, 51.0, 44.0, 45.0],   # 45 (하락)
    [45.0, 46.0, 38.0, 40.0],   # 40 (하락)
    [40.0, 42.0, 33.0, 35.0],   # 35 (하락)
    [35.0, 37.0, 28.0, 30.0],   # 30 (하락)
    [30.0, 32.0, 24.0, 25.0],   # 25 (저점)
    [25.0, 30.0, 24.0, 28.0],   # 28 (반등)
    [28.0, 35.0, 27.0, 33.0],   # 33 (반등)
    [33.0, 40.0, 32.0, 38.0],   # 38 (반등)
    [38.0, 45.0, 37.0, 43.0],   # 43 (반등)
    [43.0, 50.0, 42.0, 48.0],   # 48 (반등)
    [48.0, 52.0, 47.0, 51.0],   # 51 (원점 복귀)
])

print(f"패턴 1 (약한 상승): {pattern1[0,3]:.1f} → {pattern1[-1,3]:.1f} ({(pattern1[-1,3]/pattern1[0,3]-1)*100:.0f}% 상승)")
print(f"패턴 2 (강한 상승): {pattern2[0,3]:.1f} → {pattern2[-1,3]:.1f} ({(pattern2[-1,3]/pattern2[0,3]-1)*100:.0f}% 상승)")
print(f"패턴 3 (V자): {pattern3[0,3]:.1f} → {pattern3[-1,3]:.1f} (중간 하락 후 복귀)")

## 4. 방법 1: MinMaxScaler (실험 1 방식)

MinMaxScaler로 3개 패턴을 정규화한다. 패턴 1과 2가 같은 모양임에도 거의 동일한 값으로 정규화되는 문제를 L2 거리로 정량화한다.

---

## 5. 방법 2: First Close 기준

첫 번째 Close 기준으로 정규화한다. 같은 모양의 패턴이 동일한 값으로 매핑되지만, 값의 범위가 [0, ∞)를 벗어나 학습이 불안정해지는 문제가 있다.

In [ ]:
def normalize_minmax(pattern):
    """방법 1: Min-Max Scaler (현재 사용 중)"""
    scaler = MinMaxScaler()
    return scaler.fit_transform(pattern)

def normalize_first_value(pattern):
    """방법 2: 첫 번째 Close 기준 정규화"""
    first_close = pattern[0, 3]
    return pattern / first_close

def normalize_first_value_with_clip(pattern):
    """
    방법 3: 첫 번째 Close 기준 + Clipping + 재스케일링
    - 첫 번째 Close를 1.0으로 설정
    - 0.5 ~ 2.0 범위로 클리핑 (±100% 변동)
    - 다시 0~1로 스케일링
    """
    first_close = pattern[0, 3]
    normalized = pattern / first_close
    clipped = np.clip(normalized, 0.5, 2.0)
    rescaled = (clipped - 0.5) / 1.5
    return rescaled

def normalize_log_base(pattern):
    """
    방법 4: Log Base 정규화 (금융 데이터 표준!)
    - log(현재가 / 첫 Close) 계산
    - 비율이 같으면 같은 값 (1→10과 100→1000이 같음)
    - 급등락에 강함 (코로나 같은 이상치 처리)
    - 그 후 Min-Max로 0~1 범위로
    """
    first_close = pattern[0, 3]
    log_normalized = np.log(pattern / first_close)  # log(P_t / P_0)
    
    # Min-Max 정규화 (0~1 범위로)
    scaler = MinMaxScaler()
    return scaler.fit_transform(log_normalized)

## 6. 방법 3: First Close + Clipping + 재스케일링

첫 Close 기준 + [0.5, 2.0] 클리핑 + 재스케일링을 적용한다. 같은 모양은 동일값, 다른 모양은 다른값을 유지하면서 모든 값이 [0, 1] 범위 안에 들어오지만, ±100% 초과 변동은 잘린다.

---

## 7. 방법 1~3 종합 비교

3가지 방법을 3×3 그리드로 나란히 비교한다.

In [ ]:
norm1_1 = normalize_minmax(pattern1.copy())
norm1_2 = normalize_minmax(pattern2.copy())
norm1_3 = normalize_minmax(pattern3.copy())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Method 1: Min-Max Scaler (Current)', fontsize=16, fontweight='bold')

axes[0].plot(norm1_1[:, 3], 'o-', linewidth=2, markersize=8, label='Weak Rise')
axes[0].set_title(f'Weak Rise\n(Range: {norm1_1[:,3].min():.3f} ~ {norm1_1[:,3].max():.3f})', fontsize=12)
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(norm1_2[:, 3], 'o-', linewidth=2, markersize=8, label='Strong Rise', color='orange')
axes[1].set_title(f'Strong Rise (Same Shape x10)\n(Range: {norm1_2[:,3].min():.3f} ~ {norm1_2[:,3].max():.3f})', fontsize=12)
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(norm1_3[:, 3], 'o-', linewidth=2, markersize=8, label='V-Shape', color='green')
axes[2].set_title(f'V-Shape (Down then Up)\n(Range: {norm1_3[:,3].min():.3f} ~ {norm1_3[:,3].max():.3f})', fontsize=12)
axes[2].set_ylim(-0.1, 1.1)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# 거리 계산
dist_1_2 = np.linalg.norm(norm1_1 - norm1_2)
dist_1_3 = np.linalg.norm(norm1_1 - norm1_3)
dist_2_3 = np.linalg.norm(norm1_2 - norm1_3)

print(f"\n【Method 1: Min-Max Scaler】")
print(f"  Weak vs Strong: {dist_1_2:.4f} {'⚠️ TOO SIMILAR!' if dist_1_2 < 0.5 else '✅ Different'}")
print(f"  Weak vs V-Shape: {dist_1_3:.4f}")
print(f"  Strong vs V-Shape: {dist_2_3:.4f}")
print(f"\n  Problem: Weak and Strong have almost IDENTICAL normalized values!")

## 8. 방법 4: Log Base — 실험 3 선택 방식

Log-Base 정규화 `log(P_t / P_0)`를 적용한다.

**이 방법을 선택한 이유:** 비율 변화가 같으면 같은 값(`log(2) = 0.693`이면 100% 상승)이 되고, 코로나 급락 같은 이상치에도 로그 스케일이 자연스럽게 대응한다. 이후 MinMaxScaler를 적용해 [0, 1] 범위로 만든다. 금융 수익률 분석의 표준 방식으로 실험 3(Log)에서 채택한다.

---

## 9. 4가지 방법 전체 비교

4가지 정규화 방법을 4×3 그리드로 한눈에 비교한다. 가로축은 3개 테스트 패턴, 세로축은 4가지 방법으로 배치해 각 방법의 특성 차이를 직관적으로 파악한다.

In [ ]:
norm2_1 = normalize_first_value(pattern1.copy())
norm2_2 = normalize_first_value(pattern2.copy())
norm2_3 = normalize_first_value(pattern3.copy())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Method 2: First Close Normalization', fontsize=16, fontweight='bold')

axes[0].plot(norm2_1[:, 3], 'o-', linewidth=2, markersize=8, label='Weak Rise')
axes[0].set_title(f'Weak Rise\n(Range: {norm2_1[:,3].min():.3f} ~ {norm2_1[:,3].max():.3f})', fontsize=12)
axes[0].set_ylim(0, 6)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(norm2_2[:, 3], 'o-', linewidth=2, markersize=8, label='Strong Rise', color='orange')
axes[1].set_title(f'Strong Rise\n(Range: {norm2_2[:,3].min():.3f} ~ {norm2_2[:,3].max():.3f})', fontsize=12)
axes[1].set_ylim(0, 6)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(norm2_3[:, 3], 'o-', linewidth=2, markersize=8, label='V-Shape', color='green')
axes[2].set_title(f'V-Shape\n(Range: {norm2_3[:,3].min():.3f} ~ {norm2_3[:,3].max():.3f})', fontsize=12)
axes[2].set_ylim(0, 6)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# 거리 계산
dist_1_2 = np.linalg.norm(norm2_1 - norm2_2)
dist_1_3 = np.linalg.norm(norm2_1 - norm2_3)
dist_2_3 = np.linalg.norm(norm2_2 - norm2_3)

print(f"\n【Method 2: First Close Normalization】")
print(f"  Weak vs Strong: {dist_1_2:.4f} {'✅ SAME SHAPE!' if dist_1_2 < 0.1 else '⚠️ Different'}")
print(f"  Weak vs V-Shape: {dist_1_3:.4f}")
print(f"  Strong vs V-Shape: {dist_2_3:.4f}")
print(f"\n  Good: Weak and Strong have IDENTICAL normalized values (same shape!)")
print(f"  Issue: Values > 1.0, not bounded to [0, 1]")

## 5. 방법 3: 첫 번째 Close + Clipping (추천!)

첫 Close + Clipping + 재스케일링으로 3개 패턴을 정규화하고 결과를 확인합니다. 같은 모양의 패턴이 동일한 값으로, 다른 모양은 다른 값으로 매핑되면서 범위도 [0, 1]로 안정적입니다.

In [ ]:
norm3_1 = normalize_first_value_with_clip(pattern1.copy())
norm3_2 = normalize_first_value_with_clip(pattern2.copy())
norm3_3 = normalize_first_value_with_clip(pattern3.copy())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Method 3: First Close + Clipping + Rescale (RECOMMENDED)', fontsize=16, fontweight='bold')

axes[0].plot(norm3_1[:, 3], 'o-', linewidth=2, markersize=8, label='Weak Rise')
axes[0].set_title(f'Weak Rise\n(Range: {norm3_1[:,3].min():.3f} ~ {norm3_1[:,3].max():.3f})', fontsize=12)
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(norm3_2[:, 3], 'o-', linewidth=2, markersize=8, label='Strong Rise', color='orange')
axes[1].set_title(f'Strong Rise\n(Range: {norm3_2[:,3].min():.3f} ~ {norm3_2[:,3].max():.3f})', fontsize=12)
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(norm3_3[:, 3], 'o-', linewidth=2, markersize=8, label='V-Shape', color='green')
axes[2].set_title(f'V-Shape\n(Range: {norm3_3[:,3].min():.3f} ~ {norm3_3[:,3].max():.3f})', fontsize=12)
axes[2].set_ylim(-0.1, 1.1)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# 거리 계산
dist_1_2 = np.linalg.norm(norm3_1 - norm3_2)
dist_1_3 = np.linalg.norm(norm3_1 - norm3_3)
dist_2_3 = np.linalg.norm(norm3_2 - norm3_3)

print(f"\n【Method 3: First Close + Clipping】")
print(f"  Weak vs Strong: {dist_1_2:.4f} {'✅ SAME SHAPE!' if dist_1_2 < 0.1 else '⚠️ Different'}")
print(f"  Weak vs V-Shape: {dist_1_3:.4f} {'✅ DIFFERENT!' if dist_1_3 > 1.0 else '⚠️ Too similar'}")
print(f"  Strong vs V-Shape: {dist_2_3:.4f} {'✅ DIFFERENT!' if dist_2_3 > 1.0 else '⚠️ Too similar'}")
print(f"\n  ✅ BEST: Same shape → same values, different shape → different values!")

---

## 10. 결론: 방법별 비교표

| 방법 | 같은 모양 인식 | 범위 | 급등락 처리 | 선택 여부 |
|---|---|---|---|---|
| 1. MinMaxScaler | 불가 | [0, 1] | 불가 | 실험 1 (채택) |
| 2. First Close | 가능 | [0, ∞) | 불가 | 미채택 |
| 3. Clipping | 가능 | [0, 1] | 부분 | 미채택 |
| 4. Log Base | 가능 | [0, 1] | 가능 | 실험 3 (미채택) |

**최종 결론:** 이론적으로는 Log Base가 우수하지만, 실제 학습 결과(07 노트북)에서 실험 1(MinMaxScaler)이 **시각적 유사도 측면에서 더 나은 검색 결과**를 보여 실험 1이 최종 채택되었다.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
fig.suptitle('All Normalization Methods Comparison', fontsize=18, fontweight='bold')

# 방법 1: Min-Max
axes[0, 0].plot(norm1_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[0, 0].set_title('Method 1: Weak Rise', fontsize=11)
axes[0, 0].set_ylim(-0.1, 1.1)
axes[0, 0].set_ylabel('Method 1\nMin-Max', fontsize=10, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(norm1_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[0, 1].set_title('Method 1: Strong Rise', fontsize=11)
axes[0, 1].set_ylim(-0.1, 1.1)
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(norm1_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[0, 2].set_title('Method 1: V-Shape', fontsize=11)
axes[0, 2].set_ylim(-0.1, 1.1)
axes[0, 2].grid(alpha=0.3)

# 방법 2: First Close
axes[1, 0].plot(norm2_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[1, 0].set_title('Method 2: Weak Rise', fontsize=11)
axes[1, 0].set_ylim(0, 6)
axes[1, 0].set_ylabel('Method 2\nFirst Close', fontsize=10, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(norm2_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[1, 1].set_title('Method 2: Strong Rise', fontsize=11)
axes[1, 1].set_ylim(0, 6)
axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(norm2_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[1, 2].set_title('Method 2: V-Shape', fontsize=11)
axes[1, 2].set_ylim(0, 6)
axes[1, 2].grid(alpha=0.3)

# 방법 3: Clipping
axes[2, 0].plot(norm3_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[2, 0].set_title('Method 3: Weak Rise', fontsize=11)
axes[2, 0].set_ylim(-0.1, 1.1)
axes[2, 0].set_ylabel('Method 3\nClipping', fontsize=10, fontweight='bold')
axes[2, 0].grid(alpha=0.3)

axes[2, 1].plot(norm3_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[2, 1].set_title('Method 3: Strong Rise', fontsize=11)
axes[2, 1].set_ylim(-0.1, 1.1)
axes[2, 1].grid(alpha=0.3)

axes[2, 2].plot(norm3_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[2, 2].set_title('Method 3: V-Shape', fontsize=11)
axes[2, 2].set_ylim(-0.1, 1.1)
axes[2, 2].grid(alpha=0.3)

# 방법 4: Log Base ⭐
axes[3, 0].plot(norm4_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[3, 0].set_title('Method 4: Weak Rise ⭐', fontsize=11)
axes[3, 0].set_ylim(-0.1, 1.1)
axes[3, 0].set_ylabel('Method 4\nLog Base', fontsize=10, fontweight='bold')
axes[3, 0].set_xlabel('Week', fontsize=10)
axes[3, 0].grid(alpha=0.3)

axes[3, 1].plot(norm4_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[3, 1].set_title('Method 4: Strong Rise ⭐', fontsize=11)
axes[3, 1].set_ylim(-0.1, 1.1)
axes[3, 1].set_xlabel('Week', fontsize=10)
axes[3, 1].grid(alpha=0.3)

axes[3, 2].plot(norm4_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[3, 2].set_title('Method 4: V-Shape ⭐', fontsize=11)
axes[3, 2].set_ylim(-0.1, 1.1)
axes[3, 2].set_xlabel('Week', fontsize=10)
axes[3, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 결론 및 추천

### ⚠️ 방법 1 (Min-Max Scaler) - 현재 사용 중
- **문제점**: 약한 상승과 강한 상승이 거의 같은 값으로 정규화됨
- **결과**: Top-10에 같은 종목이 많이 나타남 (평균 3-4개)

### ✅ 방법 2 (첫 번째 Close 기준)
- **장점**: 같은 모양이면 같은 값으로 정규화됨
- **단점**: 값이 [0, 1] 범위를 벗어남 (학습 불안정)

### ⭐ 방법 3 (첫 번째 Close + Clipping)
- **장점 1**: 같은 모양이면 같은 값
- **장점 2**: 모든 값이 [0, 1] 범위 (학습 안정)
- **장점 3**: 변동폭 반영
- **단점**: ±100% 이상 변동은 잘림

### 🚀 방법 4 (Log Base) - **최고 추천!**
- **장점 1**: 금융 데이터 표준 방식 (Log Return)
- **장점 2**: 같은 비율 = 같은 값 (1→10과 100→1000이 같음)
- **장점 3**: 급등락에 강함 (코로나 같은 이상치 처리)
- **장점 4**: 대칭적 (50% 하락 = -0.69, 100% 상승 = 0.69)
- **장점 5**: [0, 1] 범위 (학습 안정)
- **추천 이유**: 실제 금융 데이터 분석에서 가장 많이 쓰임!

### 📊 비교표

| 방법 | 같은 모양 인식 | 범위 | 급등락 처리 | 추천도 |
|------|---------------|------|-----------|-------|
| 1. Min-Max | ❌ | [0, 1] | ❌ | ⭐ |
| 2. First Close | ✅ | [0, ∞] | ❌ | ⭐⭐ |
| 3. Clipping | ✅ | [0, 1] | △ | ⭐⭐⭐ |
| 4. Log Base | ✅ | [0, 1] | ✅ | ⭐⭐⭐⭐⭐ |

### 다음 단계
1. **patron_전처리.ipynb를 방법 4 (Log Base)로 변경** ← 최우선!
2. 재학습 (patron_03 또는 patron_07)
3. Top-10 같은 종목 개수 확인 (평균 1-2개 목표)
4. 실제 패턴 검색 테스트 (낙타모양, 기영이 머리 등)

Log-Base 정규화(`log(P_t / P_0)`)를 계산하고 MinMaxScaler를 적용합니다. 비율 변화가 같으면 같은 값, 다른 모양은 다른 값으로 매핑되며 급등락에도 강합니다.

In [ ]:
norm4_1 = normalize_log_base(pattern1.copy())
norm4_2 = normalize_log_base(pattern2.copy())
norm4_3 = normalize_log_base(pattern3.copy())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Method 4: Log Base (Financial Standard!)', fontsize=16, fontweight='bold')

axes[0].plot(norm4_1[:, 3], 'o-', linewidth=2, markersize=8, label='Weak Rise')
axes[0].set_title(f'Weak Rise\n(Range: {norm4_1[:,3].min():.3f} ~ {norm4_1[:,3].max():.3f})', fontsize=12)
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(norm4_2[:, 3], 'o-', linewidth=2, markersize=8, label='Strong Rise', color='orange')
axes[1].set_title(f'Strong Rise\n(Range: {norm4_2[:,3].min():.3f} ~ {norm4_2[:,3].max():.3f})', fontsize=12)
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(norm4_3[:, 3], 'o-', linewidth=2, markersize=8, label='V-Shape', color='green')
axes[2].set_title(f'V-Shape\n(Range: {norm4_3[:,3].min():.3f} ~ {norm4_3[:,3].max():.3f})', fontsize=12)
axes[2].set_ylim(-0.1, 1.1)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# 거리 계산
dist_1_2 = np.linalg.norm(norm4_1 - norm4_2)
dist_1_3 = np.linalg.norm(norm4_1 - norm4_3)
dist_2_3 = np.linalg.norm(norm4_2 - norm4_3)

print(f"\n【Method 4: Log Base】")
print(f"  Weak vs Strong: {dist_1_2:.4f} {'✅ SAME SHAPE!' if dist_1_2 < 0.1 else '⚠️ Different'}")
print(f"  Weak vs V-Shape: {dist_1_3:.4f} {'✅ DIFFERENT!' if dist_1_3 > 1.0 else '⚠️ Too similar'}")
print(f"  Strong vs V-Shape: {dist_2_3:.4f} {'✅ DIFFERENT!' if dist_2_3 > 1.0 else '⚠️ Too similar'}")
print(f"\n  ✅ BEST for finance: Handles extreme moves (COVID) + Same ratio = Same value!")

## 6. 방법 4: Log Base (금융 표준!)

방법 1~3의 3×3 비교 그리드를 그립니다. Log-Base 방법이 추가되기 전 단계의 중간 비교로, MinMaxScaler·First-Close·Clipping 세 방법의 차이를 한눈에 확인합니다.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Normalization Methods Comparison', fontsize=18, fontweight='bold')

# 방법 1
axes[0, 0].plot(norm1_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[0, 0].set_title('Method 1: Weak Rise', fontsize=11)
axes[0, 0].set_ylim(-0.1, 1.1)
axes[0, 0].set_ylabel('Method 1\nMin-Max', fontsize=10, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(norm1_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[0, 1].set_title('Method 1: Strong Rise', fontsize=11)
axes[0, 1].set_ylim(-0.1, 1.1)
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(norm1_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[0, 2].set_title('Method 1: V-Shape', fontsize=11)
axes[0, 2].set_ylim(-0.1, 1.1)
axes[0, 2].grid(alpha=0.3)

# 방법 2
axes[1, 0].plot(norm2_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[1, 0].set_title('Method 2: Weak Rise', fontsize=11)
axes[1, 0].set_ylim(0, 6)
axes[1, 0].set_ylabel('Method 2\nFirst Close', fontsize=10, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(norm2_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[1, 1].set_title('Method 2: Strong Rise', fontsize=11)
axes[1, 1].set_ylim(0, 6)
axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(norm2_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[1, 2].set_title('Method 2: V-Shape', fontsize=11)
axes[1, 2].set_ylim(0, 6)
axes[1, 2].grid(alpha=0.3)

# 방법 3
axes[2, 0].plot(norm3_1[:, 3], 'o-', linewidth=2, markersize=6)
axes[2, 0].set_title('Method 3: Weak Rise', fontsize=11)
axes[2, 0].set_ylim(-0.1, 1.1)
axes[2, 0].set_ylabel('Method 3\nClipping', fontsize=10, fontweight='bold')
axes[2, 0].set_xlabel('Week', fontsize=10)
axes[2, 0].grid(alpha=0.3)

axes[2, 1].plot(norm3_2[:, 3], 'o-', linewidth=2, markersize=6, color='orange')
axes[2, 1].set_title('Method 3: Strong Rise', fontsize=11)
axes[2, 1].set_ylim(-0.1, 1.1)
axes[2, 1].set_xlabel('Week', fontsize=10)
axes[2, 1].grid(alpha=0.3)

axes[2, 2].plot(norm3_3[:, 3], 'o-', linewidth=2, markersize=6, color='green')
axes[2, 2].set_title('Method 3: V-Shape', fontsize=11)
axes[2, 2].set_ylim(-0.1, 1.1)
axes[2, 2].set_xlabel('Week', fontsize=10)
axes[2, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 결론 및 추천

### ⚠️ 방법 1 (Min-Max Scaler) - 현재 사용 중
- **문제점**: 약한 상승과 강한 상승이 거의 같은 값으로 정규화됨
- **결과**: Top-10에 같은 종목이 많이 나타남 (평균 3-4개)

### ✅ 방법 2 (첫 번째 Close 기준)
- **장점**: 같은 모양이면 같은 값으로 정규화됨
- **단점**: 값이 [0, 1] 범위를 벗어남 (학습 불안정)

### ⭐ 방법 3 (첫 번째 Close + Clipping) - **추천!**
- **장점 1**: 같은 모양이면 같은 값
- **장점 2**: 모든 값이 [0, 1] 범위 (학습 안정)
- **장점 3**: 변동폭 반영 (낙타모양, 기영이 머리 등 구분 가능)

### 다음 단계
1. `patron_전처리.ipynb`의 정규화 코드를 방법 3으로 변경
2. 재학습 (patron_03 또는 새 노트북)
3. Top-10 같은 종목 개수 확인 (평균 1-2개 목표)
4. 실제 패턴 검색 테스트